In [0]:
%pip install google-api-python-client --quiet
dbutils.library.restartPython()

In [0]:
from googleapiclient.discovery import build

api_key = dbutils.secrets.get(scope="youtube-scope", key="youtube-api-key")

youtube = build("youtube", "v3", developerKey=api_key)

In [0]:
channel_id = "UCEGvwrlFT1Ml4VJASkA9rMA"  # Example

response = youtube.channels().list(
    part="contentDetails",
    id=channel_id
).execute()

uploads_playlist_id = response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
print("Uploads playlist ID:", uploads_playlist_id)

In [0]:
videos = []
nextPageToken = None

while True:
    pl_response = youtube.playlistItems().list(
        part="snippet,contentDetails",
        playlistId=uploads_playlist_id,
        maxResults=50,
        pageToken=nextPageToken
    ).execute()

    videos.extend(pl_response["items"])
    nextPageToken = pl_response.get("nextPageToken")
    if not nextPageToken:
        break

print(f"Fetched {len(videos)} videos")

In [0]:
import json
from pyspark.sql.functions import from_json, schema_of_json, col, lit, current_timestamp

json_rows = [(json.dumps(v),) for v in videos]
df_videos = spark.createDataFrame(json_rows, ["value"])
schema = schema_of_json(lit(json.dumps(videos[0])))
df_videos = df_videos.select(from_json(col("value"), schema).alias("data")).select("data.*")
df_videos = df_videos.withColumn("ingestion_time", current_timestamp())
df_videos.display()